# HealthGap India — District Healthcare & Health Risk Analysis

## Objective
Identify and prioritize Indian districts with gaps in healthcare access and poor health outcomes using district-level NFHS-5 indicators such as healthcare, nutrition, maternal health, immunization and so on...

## Tools
- Python
- Pandas
- NumPy
- Power Query
- Power BI

## Analytical workflow
Data Profiling → Cleaning → Indicator Selection → EDA →
KPI Framework → HealthGap Score → District Ranking →
Power Query → Power BI → Insights → Recommendations

# 1. PROJECT SETUP

In [1]:
import pandas as pd
import numpy as np
import re

from google.colab import drive

drive.mount("/content/drive")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

FILE_PATH = (
    "/content/drive/MyDrive/"
    "HealthGap-District Healthcare & Health Risk Analysis/"
    "HealthGap.csv"
)

df = pd.read_csv(FILE_PATH)

GEO_COLUMNS = ["District Names", "State/UT"]

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Mounted at /content/drive
Dataset loaded successfully.
Shape: (706, 109)


# 2. Initial Data Profiling

Validate dataset structure, data types, geographic coverage,
duplicates and basic completeness before cleaning.

In [2]:
print("=" * 65)
print("INITIAL DATA PROFILE")
print("=" * 65)

print("Rows              :", len(df))
print("Columns           :", df.shape[1])
print("Duplicate rows    :", df.duplicated().sum())
print("Duplicate columns :", df.columns.duplicated().sum())
print("Missing cells     :", df.isna().sum().sum())

print("\nData types:")
print(df.dtypes.value_counts())

print("\nGeographic coverage:")
print("Unique districts  :", df["District Names"].nunique())
print("Unique states/UTs :", df["State/UT"].nunique())

print("\nDuplicate State + District:",
      df.duplicated(GEO_COLUMNS).sum())

INITIAL DATA PROFILE
Rows              : 706
Columns           : 109
Duplicate rows    : 0
Duplicate columns : 0
Missing cells     : 0

Data types:
float64    54
object     51
int64       4
Name: count, dtype: int64

Geographic coverage:
Unique districts  : 698
Unique states/UTs : 36

Duplicate State + District: 0


# 3. Special-Value & Geographic Validation

The source uses:
- `*` for suppressed/unavailable values
- `(value)` for low-sample observations

Parentheses inside district names are preserved because they are
legitimate geographic names.

In [3]:
indicator_source_columns = df.columns.difference(GEO_COLUMNS).tolist()

special_summary = []

for col in indicator_source_columns:
    s = df[col].astype("string").str.strip()

    stars = s.eq("*")
    low_sample = s.str.fullmatch(
        r"\(\s*-?\d+(?:\.\d+)?\s*\)",
        na=False
    )

    special_summary.append({
        "indicator": col,
        "star_count": stars.sum(),
        "parentheses_count": low_sample.sum()
    })

special_summary = pd.DataFrame(special_summary)

special_summary["total_special_values"] = (
    special_summary["star_count"]
    + special_summary["parentheses_count"]
)

special_summary["special_value_pct"] = (
    special_summary["total_special_values"]
    / len(df) * 100
).round(2)

special_summary = (
    special_summary[
        special_summary["total_special_values"] > 0
    ]
    .sort_values("total_special_values", ascending=False)
    .reset_index(drop=True)
)

print("Total '*' cells:",
      special_summary["star_count"].sum())

print("Total parenthesized numeric cells:",
      special_summary["parentheses_count"].sum())

display(special_summary)

Total '*' cells: 4125
Total parenthesized numeric cells: 5068


,indicator,star_count,parentheses_count,total_special_values,special_value_pct
0,Children age 6-8 months receiving solid or semi-solid food and breastmilk16 (%),642,62,704,99.72
1,"Non-breastfeeding children age 6-23 months receiving an adequate diet16, 17 (%)",643,59,702,99.43
2,Children with diarrhoea in the 2 weeks preceding the survey who received zinc (Children under ag...,492,162,654,92.63
3,Children swith diarrhoea in the 2 weeks preceding the survey taken to a health facility or healt...,492,162,654,92.63
4,Children with diarrhoea in the 2 weeks preceding the survey who received oral rehydration salts ...,492,162,654,92.63
5,Children under age 6 months exclusively breastfed16 (%),261,347,608,86.12
6,Children born at home who were taken to a health facility for a check-up within 24 hours of birt...,422,141,563,79.75
7,Pregnant women age 15-49 years who are anaemic (<11.0 g/dl)22 (%),134,421,555,78.61
8,Children with fever or symptoms of ARI in the 2 weeks preceding the survey taken to a health fac...,224,264,488,69.12
9,Children age 12-23 months fully vaccinated based on information from vaccination card only12 (%),22,323,345,48.87


In [4]:
district_parentheses = df.loc[
    df["District Names"]
    .astype("string")
    .str.contains(r"\(", na=False),
    GEO_COLUMNS
].drop_duplicates()

print("District names containing parentheses:")
display(district_parentheses)

District names containing parentheses:


,District Names,State/UT
99,Kaimur (Bhabua),Bihar
295,Leh(Ladakh),Ladakh
319,Khargone (West Nimar),Madhya Pradesh
345,Khandwa (East Nimar),Madhya Pradesh
659,Sant Ravidas Nagar (Bhadohi),Uttar Pradesh


# 4. Data Cleaning

### Cleaning rules
1. Standardize column names.
2. Preserve geographic names as text.
3. Convert `*` to missing values.
4. Remove parentheses from numeric low-sample values.
5. Record low-sample observations separately.
6. Convert analytical indicators to numeric.

In [5]:
def clean_column_name(col):
    col = str(col).strip().lower()
    col = col.replace("%", "pct")
    col = col.replace("&", "and")
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")


original_columns = df.columns.tolist()
cleaned_columns = [clean_column_name(c) for c in original_columns]

# Resolve accidental name collisions
counts = {}
unique_columns = []

for col in cleaned_columns:
    counts[col] = counts.get(col, 0) + 1
    unique_columns.append(
        col if counts[col] == 1
        else f"{col}_{counts[col]}"
    )

column_mapping = pd.DataFrame({
    "original_column": original_columns,
    "cleaned_column": unique_columns
})

df.columns = unique_columns

GEO_COLUMNS = ["district_names", "state_ut"]

indicator_columns = [
    c for c in df.columns
    if c not in GEO_COLUMNS
]

print("Columns before:", len(original_columns))
print("Columns after :", len(df.columns))
print("Duplicate columns:", df.columns.duplicated().sum())

Columns before: 109
Columns after : 109
Duplicate columns: 0


In [6]:
low_sample_flags = pd.DataFrame(
    False,
    index=df.index,
    columns=indicator_columns
)

for col in indicator_columns:

    s = df[col].astype("string").str.strip()

    low_sample_flags[col] = s.str.fullmatch(
        r"\(\s*-?\d+(?:\.\d+)?\s*\)",
        na=False
    )

    s = s.str.replace(
        r"^\(\s*(-?\d+(?:\.\d+)?)\s*\)$",
        r"\1",
        regex=True
    )

    df[col] = pd.to_numeric(
        s.replace("*", np.nan),
        errors="coerce"
    )

df["district_names"] = (
    df["district_names"].astype("string").str.strip()
)

df["state_ut"] = (
    df["state_ut"].astype("string").str.strip()
)

print("Special-value cleaning completed.")

Special-value cleaning completed.


# 5. Cleaning Validation

Confirm that encoded indicator values have been handled correctly
while legitimate district names remain unchanged.

In [7]:
remaining_stars = sum(
    df[c].astype("string").eq("*").sum()
    for c in indicator_columns
)

remaining_parenthesized = sum(
    df[c].astype("string").str.fullmatch(
        r"\(\s*-?\d+(?:\.\d+)?\s*\)",
        na=False
    ).sum()
    for c in indicator_columns
)

print("=" * 65)
print("CLEANING VALIDATION")
print("=" * 65)

print("Rows              :", len(df))
print("Columns           :", df.shape[1])
print("Duplicate rows    :", df.duplicated().sum())
print("Duplicate columns :", df.columns.duplicated().sum())
print("Remaining '*'     :", remaining_stars)
print("Remaining '(value)':", remaining_parenthesized)
print("Duplicate geography:",
      df.duplicated(GEO_COLUMNS).sum())

CLEANING VALIDATION
Rows              : 706
Columns           : 109
Duplicate rows    : 0
Duplicate columns : 0
Remaining '*'     : 0
Remaining '(value)': 0
Duplicate geography: 0


# 6. Indicator Quality Assessment

Quality is based on the percentage of districts with usable
numeric observations.

Thresholds:
- High: ≥ 90%
- Moderate: 70–89.99%
- Low: < 70%

In [8]:
indicator_master = pd.DataFrame({
    "indicator": indicator_columns,
    "total_rows": len(df)
})

indicator_master["valid_values"] = (
    df[indicator_columns].notna().sum().values
)

indicator_master["low_sample_values"] = (
    low_sample_flags.sum().values
)

indicator_master["suppressed_values"] = (
    len(df)
    - indicator_master["valid_values"]
    - indicator_master["low_sample_values"]
)

indicator_master["missing_values"] = (
    indicator_master["suppressed_values"]
    + indicator_master["low_sample_values"]
)

indicator_master["valid_percentage"] = (
    indicator_master["valid_values"]
    / len(df) * 100
)

indicator_master["quality_category"] = pd.cut(
    indicator_master["valid_percentage"],
    bins=[-np.inf, 70, 90, np.inf],
    labels=["Low", "Moderate", "High"],
    right=False
)

print(indicator_master["quality_category"].value_counts())

quality_category
High        97
Low          8
Moderate     2
Name: count, dtype: int64


# 7. Healthcare Categories & Business Indicator Selection

Indicators are categorized according to their analytical meaning.

Core indicators focus on the primary HealthGap dimensions:
- Maternal & Newborn Health
- Child Immunization
- Child Nutrition
- Child Illness & Healthcare Access

Other indicators are retained as supporting context.

Indicators with <70% usable coverage are excluded from scoring.

In [9]:
def assign_category(indicator):

    x = indicator.lower()

    explicit = {
        "women_age_20_24_years_married_before_age_18_years_pct":
            "Maternal & Newborn Health",

        "women_age_15_24_years_who_use_hygienic_methods_of_protection_during_their_menstrual_period5_pct":
            "Maternal & Newborn Health",

        "children_age_6_8_months_receiving_solid_or_semi_solid_food_and_breastmilk16_pct":
            "Child Nutrition"
    }

    if x in explicit:
        return explicit[x]

    rules = {
        "Maternal & Newborn Health": [
            "antenatal", "pregnant", "pregnancy", "delivery",
            "birth", "caesarean", "postnatal", "maternal",
            "mother", "tetanus", "newborn"
        ],

        "Child Immunization": [
            "vaccin", "bcg", "polio", "measles",
            "rotavirus", "penta", "dpt"
        ],

        "Child Nutrition": [
            "breastfed", "breastfeeding", "adequate_diet",
            "stunted", "stunting", "wasted", "wasting",
            "underweight", "overweight", "anaemic",
            "anaemia", "vitamin_a", "nutrition"
        ],

        "Child Illness & Healthcare Access": [
            "diarrhoea", "diarrhea", "fever", "ari",
            "oral_rehydration", "ors", "zinc",
            "health_facility", "health_provider"
        ],

        "Family Planning & Reproductive Health": [
            "family_planning", "contraceptive",
            "unmet_need", "fertility"
        ],

        "Adult Health & Risk Factors": [
            "tobacco", "alcohol", "blood_pressure",
            "blood_sugar", "bmi", "waist", "cancer"
        ],

        "Socioeconomic & Living Conditions": [
            "electricity", "drinking_water", "sanitation",
            "clean_fuel", "iodized_salt",
            "health_insurance", "households"
        ],

        "Education": [
            "school", "education", "pre_primary", "literate"
        ],

        "Mortality & Registration": [
            "death", "mortality", "registered"
        ],

        "Population & Survey Profile": [
            "number_of_households", "number_of_women",
            "number_of_men", "population", "sex_ratio"
        ]
    }

    for category, keywords in rules.items():
        if any(k in x for k in keywords):
            return category

    return "Other"


indicator_master["category"] = (
    indicator_master["indicator"].map(assign_category)
)

print(indicator_master["category"].value_counts())

print("\nUncategorized indicators:")
display(
    indicator_master.loc[
        indicator_master["category"].eq("Other"),
        ["indicator"]
    ]
)

category
Maternal & Newborn Health                27
Adult Health & Risk Factors              21
Child Nutrition                          15
Family Planning & Reproductive Health    12
Child Immunization                       10
Socioeconomic & Living Conditions         7
Child Illness & Healthcare Access         6
Population & Survey Profile               4
Education                                 4
Mortality & Registration                  1
Name: count, dtype: int64

Uncategorized indicators:


,indicator


In [10]:
core_categories = [
    "Maternal & Newborn Health",
    "Child Immunization",
    "Child Nutrition",
    "Child Illness & Healthcare Access"
]

indicator_master["selection"] = np.select(
    [
        indicator_master["valid_percentage"] < 70,
        indicator_master["category"].isin(core_categories)
    ],
    [
        "Exclude",
        "Core"
    ],
    default="Supporting"
)

indicator_master["selection_reason"] = np.select(
    [
        indicator_master["selection"].eq("Exclude"),
        indicator_master["selection"].eq("Core")
    ],
    [
        "Low data coverage",
        "Primary healthcare gap indicator"
    ],
    default="Supporting/contextual indicator"
)

core_indicators = indicator_master.loc[
    indicator_master["selection"].eq("Core"),
    "indicator"
].tolist()

supporting_indicators = indicator_master.loc[
    indicator_master["selection"].eq("Supporting"),
    "indicator"
].tolist()

excluded_indicators = indicator_master.loc[
    indicator_master["selection"].eq("Exclude"),
    "indicator"
].tolist()

print("Core       :", len(core_indicators))
print("Supporting :", len(supporting_indicators))
print("Excluded   :", len(excluded_indicators))

display(
    indicator_master.groupby(
        ["selection", "category"]
    ).size().reset_index(name="indicator_count")
)

Core       : 50
Supporting : 49
Excluded   : 8


,selection,category,indicator_count
0,Core,Child Illness & Healthcare Access,2
1,Core,Child Immunization,10
2,Core,Child Nutrition,12
3,Core,Maternal & Newborn Health,26
4,Exclude,Child Illness & Healthcare Access,4
5,Exclude,Child Nutrition,3
6,Exclude,Maternal & Newborn Health,1
7,Supporting,Adult Health & Risk Factors,21
8,Supporting,Education,4
9,Supporting,Family Planning & Reproductive Health,12


# 8. Final Analytical Datasets

Create:
1. Core dataset — indicators used for HealthGap scoring.
2. Supporting dataset — contextual indicators.
3. Indicator master — metadata and business classification.

In [11]:
df_core = df[
    GEO_COLUMNS + core_indicators
].copy()

df_supporting = df[
    GEO_COLUMNS + supporting_indicators
].copy()

data_dictionary = indicator_master[
    [
        "indicator",
        "category",
        "quality_category",
        "valid_percentage",
        "selection",
        "selection_reason"
    ]
].copy()

print("Core       :", df_core.shape)
print("Supporting :", df_supporting.shape)
print("Master     :", indicator_master.shape)

Core       : (706, 52)
Supporting : (706, 51)
Master     : (107, 11)


In [12]:
geo_key = (
    df_core["state_ut"].astype("string")
    + " | "
    + df_core["district_names"].astype("string")
)

assert df_core.duplicated().sum() == 0
assert df_core.columns.duplicated().sum() == 0
assert geo_key.duplicated().sum() == 0

print("=" * 65)
print("FINAL DATASET VALIDATION")
print("=" * 65)

print("Rows                    :", len(df_core))
print("Columns                 :", df_core.shape[1])
print("Core indicators         :", len(core_indicators))
print("Duplicate rows          :", df_core.duplicated().sum())
print("Duplicate geographic keys:", geo_key.duplicated().sum())
print("Missing indicator cells :",
      df_core[core_indicators].isna().sum().sum())

FINAL DATASET VALIDATION
Rows                    : 706
Columns                 : 52
Core indicators         : 50
Duplicate rows          : 0
Duplicate geographic keys: 0
Missing indicator cells : 451


# 9. Exploratory Data Analysis

EDA focuses on:
- Data completeness
- Indicator distributions
- Variability
- Category-level patterns
- Correlations
- Potential outliers
- Geographic completeness

In [13]:
indicator_summary = (
    df_core[core_indicators]
    .agg(["mean", "median", "std", "min", "max"])
    .T
)

indicator_summary["missing_pct"] = (
    df_core[core_indicators]
    .isna()
    .mean()
    .mul(100)
)

indicator_summary = indicator_summary.round(2)

print("Average missingness:",
      round(indicator_summary["missing_pct"].mean(), 2), "%")

print("\nLowest average indicators:")
display(
    indicator_summary
    .sort_values("mean")
    .head(10)
)

print("\nHighest variability:")
display(
    indicator_summary
    .sort_values("std", ascending=False)
    .head(10)
)

Average missingness: 1.28 %

Lowest average indicators:


,mean,median,std,min,max,missing_pct
births_in_the_5_years_preceding_the_survey_that_are_third_or_higher_order_pct,2.05,1.8,1.60,0.0,8.0,0.14
children_prevalence_of_symptoms_of_acute_respiratory_infection_ari_in_the_2_weeks_preceding_the_survey_children_under_age_5_years_pct,2.54,2.1,2.06,0.0,11.2,0.00
home_births_that_were_conducted_by_skilled_health_personnel_in_the_5_years_before_the_survey_10_pct,3.08,2.1,3.17,0.0,19.9,0.00
children_age_12_23_months_who_received_most_of_their_vaccinations_in_a_private_health_facility_pct,3.21,1.6,4.50,0.0,32.9,2.27
children_under_5_years_who_are_overweight_weight_for_height_20_pct,4.21,3.5,3.12,0.0,21.1,0.00
women_age_15_19_years_who_were_already_mothers_or_pregnant_at_the_time_of_the_survey_pct,6.17,4.8,5.00,0.0,27.3,0.00
prevalence_of_diarrhoea_in_the_2_weeks_preceding_the_survey_children_under_age_5_years_pct,6.48,5.5,4.10,0.0,39.3,0.00
children_under_5_years_who_are_severely_wasted_weight_for_height_19_pct,7.57,6.9,3.93,0.5,30.5,0.00
breastfeeding_children_age_6_23_months_receiving_an_adequate_diet16_17_pct,11.92,10.2,7.72,0.0,54.1,0.71
total_children_age_6_23_months_receiving_an_adequate_diet16_17_pct,12.30,10.9,7.45,0.0,50.1,0.14



Highest variability:


,mean,median,std,min,max,missing_pct
average_out_of_pocket_expenditure_per_delivery_in_a_public_health_facility_for_last_birth_in_the_5_years_before_the_survey_rs,3529.35,2830.00,2525.69,193.0,20101.0,0.14
sex_ratio_at_birth_for_children_born_in_the_last_five_years_females_per_1_000_males,944.80,932.50,121.62,658.0,1485.0,0.00
children_age_12_23_months_who_have_received_3_doses_of_rotavirus_vaccine14_pct,39.13,39.90,32.18,0.0,100.0,1.84
mothers_who_consumed_iron_folic_acid_for_100_days_or_more_when_they_were_pregnant_for_last_birth_in_the_5_years_before_the_survey_pct,46.36,47.80,21.16,1.6,95.0,0.00
mothers_who_had_at_least_4_antenatal_care_visits_for_last_birth_in_the_5_years_before_the_survey_pct,60.47,62.40,20.27,4.4,98.7,0.00
births_in_a_private_health_facility_that_were_delivered_by_caesarean_section_in_the_5_years_before_the_survey_pct,49.43,46.10,18.63,6.3,94.2,21.25
mothers_who_consumed_iron_folic_acid_for_180_days_or_more_when_they_were_pregnant_for_last_birth_in_the_5_years_before_the_survey_pct,27.32,24.55,17.90,0.8,84.6,0.00
institutional_births_in_public_facility_in_the_5_years_before_the_survey_pct,64.94,66.50,16.32,18.1,96.7,0.00
children_under_age_3_years_breastfed_within_one_hour_of_birth15_pct,44.84,45.15,16.27,7.8,88.5,0.00
births_delivered_by_caesarean_section_in_the_5_years_before_the_survey_pct,22.84,18.60,15.99,1.4,82.4,0.00


In [14]:
category_summary = (
    indicator_master[
        indicator_master["selection"].eq("Core")
    ][["indicator", "category"]]
    .merge(
        indicator_summary.reset_index()
        .rename(columns={"index": "indicator"}),
        on="indicator"
    )
    .groupby("category")
    .agg(
        indicators=("indicator", "count"),
        avg_value=("mean", "mean"),
        avg_missing_pct=("missing_pct", "mean")
    )
    .sort_values("indicators", ascending=False)
    .round(2)
)

display(category_summary)

,indicators,avg_value,avg_missing_pct
category,,,
Maternal & Newborn Health,26,222.63,1.63
Child Nutrition,12,32.73,0.08
Child Immunization,10,69.09,2.05
Child Illness & Healthcare Access,2,4.51,0.00


In [15]:
corr = df_core[core_indicators].corr()

corr_pairs = (
    corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )
    .stack()
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

print("Strongest correlations:")
display(corr_pairs.head(10))

Strongest correlations:


,,0
non_pregnant_women_age_15_49_years_who_are_anaemic_12_0_g_dl_22_pct,all_women_age_15_49_years_who_are_anaemic22_pct,0.999303
children_age_12_23_months_fully_vaccinated_based_on_information_from_either_vaccination_card_or_mother_s_recall11_pct,children_age_12_23_months_who_have_received_3_doses_of_polio_vaccine13_pct,0.965512
breastfeeding_children_age_6_23_months_receiving_an_adequate_diet16_17_pct,total_children_age_6_23_months_receiving_an_adequate_diet16_17_pct,0.962458
mothers_who_received_postnatal_care_from_a_doctor_nurse_lhv_anm_midwife_other_health_personnel_within_2_days_of_delivery_for_last_birth_in_the_5_years_before_the_survey_pct,children_who_received_postnatal_care_from_a_doctor_nurse_lhv_anm_midwife_other_health_personnel_within_2_days_of_delivery_for_last_birth_in_the_5_years_before_the_survey_pct,0.946042
institutional_births_in_the_5_years_before_the_survey_pct,births_attended_by_skilled_health_personnel_in_the_5_years_before_the_survey_10_pct,0.945676
births_delivered_by_caesarean_section_in_the_5_years_before_the_survey_pct,births_in_a_public_health_facility_that_were_delivered_by_caesarean_section_in_the_5_years_before_the_survey_pct,0.944710
children_age_12_23_months_who_have_received_3_doses_of_penta_or_dpt_vaccine_pct,children_age_12_23_months_who_have_received_3_doses_of_penta_or_hepatitis_b_vaccine_pct,0.933899
mothers_who_consumed_iron_folic_acid_for_100_days_or_more_when_they_were_pregnant_for_last_birth_in_the_5_years_before_the_survey_pct,mothers_who_consumed_iron_folic_acid_for_180_days_or_more_when_they_were_pregnant_for_last_birth_in_the_5_years_before_the_survey_pct,0.910052
non_pregnant_women_age_15_49_years_who_are_anaemic_12_0_g_dl_22_pct,all_women_age_15_19_years_who_are_anaemic22_pct,0.908927
children_age_12_23_months_who_have_received_3_doses_of_penta_or_dpt_vaccine_pct,children_age_12_23_months_who_have_received_the_first_dose_of_measles_containing_vaccine_mcv_pct,0.907451


In [16]:
q1 = df_core[core_indicators].quantile(0.25)
q3 = df_core[core_indicators].quantile(0.75)
iqr = q3 - q1

outlier_counts = (
    (
        (df_core[core_indicators] < q1 - 1.5 * iqr) |
        (df_core[core_indicators] > q3 + 1.5 * iqr)
    )
    .sum()
    .sort_values(ascending=False)
)

print("Potential statistical outliers:")
display(outlier_counts.head(15))

Potential statistical outliers:


,0
registered_pregnancies_for_which_the_mother_received_a_mother_and_child_protection_mcp_card_for_last_birth_in_the_5_years_before_the_survey_pct,48
mothers_whose_last_birth_was_protected_against_neonatal_tetanus_for_last_birth_in_the_5_years_before_the_survey_9_pct,41
average_out_of_pocket_expenditure_per_delivery_in_a_public_health_facility_for_last_birth_in_the_5_years_before_the_survey_rs,41
children_age_12_23_months_who_received_most_of_their_vaccinations_in_a_private_health_facility_pct,36
children_under_age_5_years_whose_birth_was_registered_with_the_civil_authority_pct,34
children_under_5_years_who_are_overweight_weight_for_height_20_pct,32
home_births_that_were_conducted_by_skilled_health_personnel_in_the_5_years_before_the_survey_10_pct,27
institutional_births_in_the_5_years_before_the_survey_pct,25
women_age_15_19_years_who_were_already_mothers_or_pregnant_at_the_time_of_the_survey_pct,25
children_age_12_23_months_who_received_most_of_their_vaccinations_in_a_public_health_facility_pct,24


In [17]:
state_coverage = (
    df_core.groupby("state_ut")
    .agg(
        districts=("district_names", "nunique"),
        rows=("district_names", "size")
    )
    .sort_values("districts", ascending=False)
)

district_completeness = df_core[
    GEO_COLUMNS
].copy()

district_completeness["missing_indicators"] = (
    df_core[core_indicators].isna().sum(axis=1)
)

district_completeness["valid_indicators"] = (
    df_core[core_indicators].notna().sum(axis=1)
)

print("States/UTs:", len(state_coverage))

print("\nDistricts with lowest completeness:")
display(
    district_completeness
    .sort_values("missing_indicators", ascending=False)
    .head(20)
)

States/UTs: 36

Districts with lowest completeness:


,district_names,state_ut,missing_indicators,valid_indicators
330,Jabalpur,Madhya Pradesh,16,34
323,Bhopal,Madhya Pradesh,14,36
325,Raisen,Madhya Pradesh,14,36
1,North & Middle Andaman,Andaman & Nicobar Islands,13,37
139,South Goa,Goa,12,38
287,Thrissur,Kerala,12,38
559,Tiruppur,Tamil Nadu,12,38
292,Pathanamthitta,Kerala,12,38
369,Mumbai Suburban,Maharastra,12,38
527,East District,Sikkim,12,38


# EDA Conclusion

The EDA identifies:
- Indicators with relatively low data availability.
- Indicators with high district-level variability.
- Strongly correlated indicators.
- Potential statistical outliers.
- Geographic differences in data completeness.

These findings inform the KPI framework and HealthGap scoring methodology.

# 10. HealthGap KPI Framework

Each Core indicator receives a business direction:

- Positive → higher value indicates better performance.
- Negative → higher value indicates greater health risk/gap.
- Context → retained for interpretation but excluded from scoring.

Scored indicators receive equal weights.

In [18]:
def assign_direction(indicator):

    x = indicator.lower()

    negative = [
        "diarrhoea", "diarrhea", "acute_respiratory",
        "symptoms_of_acute", "stunted", "wasted",
        "severely_wasted", "underweight", "overweight",
        "anaemic", "anaemia", "unmet_need",
        "high_risk", "tobacco", "alcohol",
        "blood_pressure", "blood_sugar",
        "bmi_is_below_normal"
    ]

    positive = [
        "vaccinated", "vaccination", "bcg", "polio",
        "measles", "rotavirus", "penta", "dpt",
        "vitamin_a", "adequate_diet", "breastfed",
        "antenatal", "iron_folic", "postnatal",
        "institutional_births", "skilled_health_personnel",
        "births_attended", "tetanus", "mcp_card",
        "birth_was_registered", "hygienic_methods"
    ]

    if any(k in x for k in negative):
        return "Negative"

    if any(k in x for k in positive):
        return "Positive"

    return "Context"


indicator_master["direction"] = (
    indicator_master["indicator"].map(assign_direction)
)

direction_overrides = {
    "sex_ratio_at_birth_for_children_born_in_the_last_five_years_females_per_1_000_males": "Context",
    "births_in_the_5_years_preceding_the_survey_that_are_third_or_higher_order_pct": "Context",
    "average_out_of_pocket_expenditure_per_delivery_in_a_public_health_facility_for_last_birth_in_the_5_years_before_the_survey_rs": "Context",
    "births_delivered_by_caesarean_section_in_the_5_years_before_the_survey_pct": "Context",
    "births_in_a_private_health_facility_that_were_delivered_by_caesarean_section_in_the_5_years_before_the_survey_pct": "Context",
    "births_in_a_public_health_facility_that_were_delivered_by_caesarean_section_in_the_5_years_before_the_survey_pct": "Context",
    "children_age_12_23_months_who_received_most_of_their_vaccinations_in_a_public_health_facility_pct": "Context",
    "children_age_12_23_months_who_received_most_of_their_vaccinations_in_a_private_health_facility_pct": "Context",

    "women_age_20_24_years_married_before_age_18_years_pct": "Negative",
    "women_age_15_19_years_who_were_already_mothers_or_pregnant_at_the_time_of_the_survey_pct": "Negative",
    "women_age_15_24_years_who_use_hygienic_methods_of_protection_during_their_menstrual_period5_pct": "Positive",
    "children_age_12_23_months_who_have_received_bcg_pct": "Positive"
}

indicator_master["direction"] = (
    indicator_master["indicator"]
    .map(direction_overrides)
    .fillna(indicator_master["direction"])
)

core_kpis = indicator_master[
    indicator_master["selection"].eq("Core")
].copy()

print(core_kpis["direction"].value_counts())

direction
Positive    27
Negative    15
Context      8
Name: count, dtype: int64


In [19]:
valid_directions = {"Positive", "Negative", "Context"}

assert core_kpis["direction"].isin(valid_directions).all()

core_kpis["score_included"] = (
    core_kpis["direction"].isin(["Positive", "Negative"])
)

scoring_count = core_kpis["score_included"].sum()

core_kpis["weight"] = np.where(
    core_kpis["score_included"],
    1 / scoring_count,
    0
)

core_kpis["scoring_method"] = np.where(
    core_kpis["score_included"],
    "Min-Max 0-100",
    "Context Only"
)

print("Core indicators :", len(core_kpis))
print("Scored indicators:", scoring_count)
print("Context indicators:",
      (~core_kpis["score_included"]).sum())
print("Weight sum:", core_kpis["weight"].sum())

Core indicators : 50
Scored indicators: 42
Context indicators: 8
Weight sum: 1.0


In [20]:
kpi_master = core_kpis[
    [
        "indicator",
        "category",
        "quality_category",
        "valid_percentage",
        "direction",
        "weight",
        "scoring_method",
        "score_included"
    ]
].sort_values(
    ["score_included", "category", "indicator"],
    ascending=[False, True, True]
).reset_index(drop=True)

kpi_master.to_csv(
    "healthgap_kpi_master.csv",
    index=False
)

assert kpi_master["indicator"].is_unique
assert kpi_master["direction"].notna().all()
assert np.isclose(kpi_master["weight"].sum(), 1)

print("KPI framework validated.")

KPI framework validated.


#11. HealthGap Score

For every scored indicator:

### Positive indicator
Higher value → higher score.

### Negative indicator
Lower value → higher score.

Scores are normalized from 0–100 using district-level
min-max normalization.

The overall district score is the mean of available indicator scores.

In [21]:
scoring_master = kpi_master[
    kpi_master["score_included"]
].copy()

scoring_indicators = scoring_master["indicator"].tolist()

score_df = df_core[
    GEO_COLUMNS + scoring_indicators
].copy()

for _, row in scoring_master.iterrows():

    col = row["indicator"]
    x = score_df[col]

    min_val = x.min()
    max_val = x.max()

    if pd.isna(min_val) or min_val == max_val:
        score_df[col] = np.nan
        continue

    if row["direction"] == "Positive":
        score_df[col] = (
            (x - min_val) /
            (max_val - min_val) * 100
        )
    else:
        score_df[col] = (
            (max_val - x) /
            (max_val - min_val) * 100
        )

score_values = score_df[scoring_indicators]

print("Scored indicators:", len(scoring_indicators))
print("Minimum score:", score_values.min().min())
print("Maximum score:", score_values.max().max())
print("Missing score cells:", score_values.isna().sum().sum())

Scored indicators: 42
Minimum score: 0.0
Maximum score: 100.0
Missing score cells: 267


#12. District HealthGap Score & Ranking

District score = average of available normalized KPI scores.

Minimum score coverage required for ranking:
80%.

Districts below the threshold remain unranked.

In [22]:
district_score = score_df[
    GEO_COLUMNS + scoring_indicators
].copy()

district_score["healthgap_score"] = (
    district_score[scoring_indicators]
    .mean(axis=1)
)

district_score["available_indicators"] = (
    district_score[scoring_indicators]
    .notna()
    .sum(axis=1)
)

district_score["score_coverage_pct"] = (
    district_score["available_indicators"]
    / len(scoring_indicators) * 100
)

MIN_COVERAGE = 80

district_score["score_status"] = np.where(
    district_score["score_coverage_pct"] >= MIN_COVERAGE,
    "Reliable",
    "Low Coverage"
)

district_score["healthgap_gap"] = (
    100 - district_score["healthgap_score"]
)

print("Districts:", len(district_score))
print(district_score["score_status"].value_counts())

Districts: 706
score_status
Reliable        693
Low Coverage     13
Name: count, dtype: int64


In [23]:
eligible = (
    district_score["score_status"].eq("Reliable")
)

ranking_df = district_score.copy()
ranking_df["district_rank"] = pd.NA

eligible_ranked = (
    ranking_df.loc[eligible]
    .sort_values(
        ["healthgap_score", "state_ut", "district_names"],
        ascending=[False, True, True]
    )
)

ranking_df.loc[
    eligible_ranked.index,
    "district_rank"
] = np.arange(1, len(eligible_ranked) + 1)

ranking_df["district_rank"] = (
    pd.to_numeric(
        ranking_df["district_rank"],
        errors="coerce"
    ).astype("Int64")
)

print("Eligible districts:", eligible.sum())
print("Unranked districts:", (~eligible).sum())
print(
    "Unique ranks:",
    ranking_df["district_rank"].dropna().nunique()
)

Eligible districts: 693
Unranked districts: 13
Unique ranks: 693


#13. State-Level HealthGap Summary

Compare average district performance across States/UTs and
measure the internal spread between the strongest and weakest
eligible districts.

In [24]:
state_summary = (
    ranking_df[eligible]
    .groupby("state_ut")
    .agg(
        districts=("district_names", "nunique"),
        avg_healthgap_score=("healthgap_score", "mean"),
        avg_healthgap_gap=("healthgap_gap", "mean"),
        best_district_score=("healthgap_score", "max"),
        worst_district_score=("healthgap_score", "min")
    )
)

state_summary["district_score_gap"] = (
    state_summary["best_district_score"]
    - state_summary["worst_district_score"]
)

state_summary = (
    state_summary
    .sort_values("avg_healthgap_score", ascending=False)
    .reset_index()
)

state_summary["state_rank"] = (
    np.arange(1, len(state_summary) + 1)
)

display(
    state_summary[
        [
            "state_rank",
            "state_ut",
            "districts",
            "avg_healthgap_score",
            "avg_healthgap_gap",
            "district_score_gap"
        ]
    ].head(10)
)

,state_rank,state_ut,districts,avg_healthgap_score,avg_healthgap_gap,district_score_gap
0,1,Puducherry,4,74.351711,25.648289,8.698056
1,2,Goa,1,73.559038,26.440962,0.0
2,3,Tamil Nadu,31,73.481985,26.518015,12.472316
3,4,Kerala,11,72.949333,27.050667,15.925165
4,5,Lakshadweep,1,72.855878,27.144122,0.0
5,6,Himachal Pradesh,12,71.517618,28.482382,9.98663
6,7,Odisha,30,70.817018,29.182982,17.99703
7,8,Sikkim,3,70.707193,29.292807,2.225542
8,9,Chandigarh,1,70.030123,29.969877,0.0
9,10,Andaman & Nicobar Islands,2,69.105241,30.894759,6.247342


#14. District Disparity Analysis

Measure each district's performance relative to its own
State/UT average.

In [25]:
disparity_df = ranking_df.loc[eligible].copy()

state_avg = (
    disparity_df
    .groupby("state_ut")["healthgap_score"]
    .transform("mean")
)

disparity_df["district_vs_state"] = (
    disparity_df["healthgap_score"] - state_avg
)

disparity_df["performance_vs_state"] = np.select(
    [
        disparity_df["district_vs_state"] > 0,
        disparity_df["district_vs_state"] < 0
    ],
    [
        "Above State Average",
        "Below State Average"
    ],
    default="At State Average"
)

print("Districts below state average:",
      (disparity_df["performance_vs_state"] ==
       "Below State Average").sum())

print("Districts above state average:",
      (disparity_df["performance_vs_state"] ==
       "Above State Average").sum())

Districts below state average: 329
Districts above state average: 361


#15. Final District Score Table

This is the main analytical output that will feed Power Query
and Power BI.

In [26]:
district_score_final = (
    ranking_df[
        [
            "state_ut",
            "district_names",
            "healthgap_score",
            "healthgap_gap",
            "district_rank",
            "score_coverage_pct",
            "score_status"
        ]
    ]
    .merge(
        disparity_df[
            [
                "state_ut",
                "district_names",
                "district_vs_state",
                "performance_vs_state"
            ]
        ],
        on=["state_ut", "district_names"],
        how="left"
    )
    .sort_values(
        "district_rank",
        na_position="last"
    )
    .reset_index(drop=True)
)

assert len(district_score_final) == len(df_core)
assert district_score_final.duplicated(
    ["state_ut", "district_names"]
).sum() == 0

print("=" * 65)
print("FINAL DISTRICT SCORE TABLE")
print("=" * 65)

print("Rows:", len(district_score_final))
print("Columns:", len(district_score_final.columns))
print("Duplicate rows:",
      district_score_final.duplicated().sum())

display(district_score_final.head(10))

FINAL DISTRICT SCORE TABLE
Rows: 706
Columns: 9
Duplicate rows: 0


,state_ut,district_names,healthgap_score,healthgap_gap,district_rank,score_coverage_pct,score_status,district_vs_state,performance_vs_state
0,Puducherry,Mahe,80.279645,19.720355,1,100.000000,Reliable,5.927933,Above State Average
1,Odisha,Puri,79.19275,20.80725,2,97.619048,Reliable,8.375732,Above State Average
2,Odisha,Jharsuguda,78.631339,21.368661,3,97.619048,Reliable,7.814321,Above State Average
3,Kerala,Kozhikode,78.199277,21.800723,4,100.000000,Reliable,5.249944,Above State Average
4,Kerala,Kasaragod,77.807056,22.192944,5,100.000000,Reliable,4.857723,Above State Average
5,Tamil Nadu,Theni,77.588504,22.411496,6,100.000000,Reliable,4.106519,Above State Average
6,Tamil Nadu,The Nilgiris,77.46027,22.53973,7,97.619048,Reliable,3.978285,Above State Average
7,Tamil Nadu,Thiruvarur,77.1195,22.8805,8,97.619048,Reliable,3.637515,Above State Average
8,Tamil Nadu,Vellore,77.098986,22.901014,9,97.619048,Reliable,3.617001,Above State Average
9,Tamil Nadu,Thoothukkudi,76.685688,23.314312,10,97.619048,Reliable,3.203703,Above State Average


# 16. Business Insights

Translate the HealthGap analysis into actionable business insights by examining:

- Districts with the largest healthcare gaps
- Districts performing below their State/UT average
- Key indicator-level relationships
- Areas requiring priority intervention

Correlation is used only to identify associations and does not imply causation.

In [27]:
GEO_COLUMNS = ["state_ut", "district_names"]

reliable = district_score_final[
    district_score_final["score_status"].eq("Reliable")
].copy()

# Priority districts

priority_districts = reliable.nlargest(
    10, "healthgap_score"
)[[
    *GEO_COLUMNS,
    "healthgap_score",
    "healthgap_gap",
    "district_rank",
    "score_coverage_pct"
]]

below_state = reliable[
    reliable["performance_vs_state"].eq("Below State Average")
].nsmallest(
    10, "district_vs_state"
)[[
    *GEO_COLUMNS,
    "healthgap_score",
    "district_vs_state",
    "district_rank",
    "score_coverage_pct"
]]

# State-level opportunity areas

state_opportunities = (
    state_summary
    .sort_values(
        ["avg_healthgap_score", "district_score_gap"],
        ascending=[True, False]
    )
    [[
        "state_rank",
        "state_ut",
        "districts",
        "avg_healthgap_score",
        "avg_healthgap_gap",
        "district_score_gap"
    ]]
)


In [28]:
# Key business relationships

relationships = {
    "Maternal Care → Institutional Births": (
        "mothers_who_had_at_least_4_antenatal_care_visits_for_last_birth_in_the_5_years_before_the_survey_pct",
        "institutional_births_in_the_5_years_before_the_survey_pct"
    ),
    "Maternal Care → Child Stunting": (
        "mothers_who_had_at_least_4_antenatal_care_visits_for_last_birth_in_the_5_years_before_the_survey_pct",
        "children_under_5_years_who_are_stunted_height_for_age_18_pct"
    )
}

relationship_rows = []

for question, (x_col, y_col) in relationships.items():
    if {x_col, y_col}.issubset(df_core.columns):
        temp = df_core[[x_col, y_col]].dropna()

        if len(temp) >= 30:
            r = temp[x_col].corr(temp[y_col])

            relationship_rows.append({
                "business_question": question,
                "observations": len(temp),
                "correlation": round(r, 3)
            })

relationship_df = pd.DataFrame(relationship_rows)

if not relationship_df.empty:
    relationship_df["strength"] = pd.cut(
        relationship_df["correlation"].abs(),
        bins=[-np.inf, 0.10, 0.30, 0.50, 0.70, np.inf],
        labels=[
            "Very Weak",
            "Weak",
            "Moderate",
            "Strong",
            "Very Strong"
        ],
        right=False
    )

    relationship_df["analytical_note"] = np.where(
        relationship_df["correlation"].abs() >= 0.50,
        "Strong association; does not establish causation.",
        "Descriptive association; does not establish causation."
    )

    relationship_df = relationship_df.sort_values(
        "correlation",
        key=lambda s: s.abs(),
        ascending=False
    ).reset_index(drop=True)


In [29]:
# Final insight summary

below_count = (
    reliable["performance_vs_state"]
    .eq("Below State Average")
    .sum()
)

above_count = (
    reliable["performance_vs_state"]
    .eq("Above State Average")
    .sum()
)

print("=" * 65)
print("BUSINESS INSIGHTS")
print("=" * 65)

print(f"Reliable districts       : {len(reliable)}")
print(f"Below State Average      : {below_count}")
print(f"Above State Average      : {above_count}")
print(f"Lowest HealthGap Score   : {reliable['healthgap_score'].min():.2f}")
print(f"Highest HealthGap Score  : {reliable['healthgap_score'].max():.2f}")
print(f"Average HealthGap Score  : {reliable['healthgap_score'].mean():.2f}")
print(f"Lowest-performing State  : {state_summary.iloc[-1]['state_ut']}")
print(f"Highest-performing State : {state_summary.iloc[0]['state_ut']}")

print("\nBOTTOM 10 — HIGHEST HEALTHGAP")
display(priority_districts)

print("\nBOTTOM 10 — MOST BELOW STATE AVERAGE")
display(below_state)

print("\nSTATE-LEVEL OPPORTUNITY AREAS")
display(state_opportunities.head(10))

print("\nKEY BUSINESS RELATIONSHIPS")
display(relationship_df)

BUSINESS INSIGHTS
Reliable districts       : 693
Below State Average      : 329
Above State Average      : 361
Lowest HealthGap Score   : 39.23
Highest HealthGap Score  : 80.28
Average HealthGap Score  : 62.10
Lowest-performing State  : Bihar
Highest-performing State : Puducherry

BOTTOM 10 — HIGHEST HEALTHGAP


,state_ut,district_names,healthgap_score,healthgap_gap,district_rank,score_coverage_pct
0,Puducherry,Mahe,80.279645,19.720355,1,100.000000
1,Odisha,Puri,79.19275,20.80725,2,97.619048
2,Odisha,Jharsuguda,78.631339,21.368661,3,97.619048
3,Kerala,Kozhikode,78.199277,21.800723,4,100.000000
4,Kerala,Kasaragod,77.807056,22.192944,5,100.000000
5,Tamil Nadu,Theni,77.588504,22.411496,6,100.000000
6,Tamil Nadu,The Nilgiris,77.46027,22.53973,7,97.619048
7,Tamil Nadu,Thiruvarur,77.1195,22.8805,8,97.619048
8,Tamil Nadu,Vellore,77.098986,22.901014,9,97.619048
9,Tamil Nadu,Thoothukkudi,76.685688,23.314312,10,97.619048



BOTTOM 10 — MOST BELOW STATE AVERAGE


,state_ut,district_names,healthgap_score,district_vs_state,district_rank,score_coverage_pct
643,Haryana,Mewat,50.011013,-16.0322,644,100.000000
683,Maharastra,Dhule,45.781579,-14.918634,684,100.000000
645,Chhattisgarh,Bastar,49.95018,-13.361781,646,100.000000
669,Maharastra,Parbhani,47.484315,-13.215897,670,100.000000
689,Jharkhand,Deoghar,42.904796,-13.202154,690,100.000000
614,Jammu & Kashmir,Doda,52.063519,-12.280335,615,100.000000
692,Nagaland,Tuensang,39.22784,-12.258368,693,100.000000
687,Meghalaya,North Garo Hills,43.08182,-12.087082,688,97.619048
656,Manipur,Ukhrul,49.068782,-11.769095,657,100.000000
684,Uttar Pradesh,Bahraich,45.177452,-11.731204,685,100.000000



STATE-LEVEL OPPORTUNITY AREAS


,state_rank,state_ut,districts,avg_healthgap_score,avg_healthgap_gap,district_score_gap
35,36,Bihar,38,49.485912,50.514088,15.195988
34,35,Nagaland,11,51.486208,48.513792,27.567265
33,34,Tripura,8,54.420261,45.579739,18.829539
32,33,Meghalaya,11,55.168902,44.831098,22.088042
31,32,Assam,33,55.995976,44.004024,17.534586
30,31,Jharkhand,24,56.106951,43.893049,22.422684
29,30,Uttar Pradesh,75,56.908656,43.091344,21.253372
28,29,Arunachal Pradesh,20,58.507641,41.492359,14.992392
27,28,Gujarat,33,59.693784,40.306216,20.146714
26,27,Maharastra,34,60.700212,39.299788,23.568945



KEY BUSINESS RELATIONSHIPS


,business_question,observations,correlation,strength,analytical_note
0,Maternal Care → Institutional Births,706,0.599,Strong,Strong association; does not establish causation.
1,Maternal Care → Child Stunting,706,-0.360,Moderate,Descriptive association; does not establish causation.


# 17. Recommendations

Translate validated findings into actionable recommendations.

Recommendations are based on:
- District HealthGap performance
- Within-State disparities
- State-level opportunity areas
- Validated indicator relationships
- Score coverage and data-quality limitations

Recommendations are analytical priorities, not causal or clinical conclusions.

In [30]:
lowest_state = state_summary.iloc[-1]["state_ut"]

recommendations = pd.DataFrame([
    {
        "priority": "High",
        "area": "District-level healthcare",
        "finding": (
            "The lowest-ranked reliable districts have substantially "
            "larger HealthGap values."
        ),
        "recommendation": (
            "Prioritize targeted healthcare interventions in "
            "the lowest-performing districts."
        ),
        "business_action": (
            "Use district ranking and HealthGap gap to prioritize "
            "locations for resource allocation."
        )
    },
    {
        "priority": "High",
        "area": "Within-state disparity",
        "finding": (
            f"{below_count} reliable districts perform below "
            "their respective State/UT average."
        ),
        "recommendation": (
            "Use district-level performance rather than state "
            "averages alone when identifying priority areas."
        ),
        "business_action": (
            "Investigate districts with the largest negative "
            "district_vs_state values."
        )
    },
    {
        "priority": "High",
        "area": lowest_state,
        "finding": (
            f"{lowest_state} has the lowest average HealthGap "
            "score among the States/UTs analyzed."
        ),
        "recommendation": (
            f"Conduct an indicator-level assessment of {lowest_state} "
            "to identify the main contributors to the observed gap."
        ),
        "business_action": (
            f"Prioritize {lowest_state} for deeper diagnostic analysis "
            "before allocating interventions."
        )
    },
    {
        "priority": "Medium",
        "area": "Maternal care",
        "finding": (
            "Maternal Care and Institutional Births show a "
            "strong positive association (r = 0.599)."
        ),
        "recommendation": (
            "Strengthen maternal-care pathways that support "
            "access to institutional delivery."
        ),
        "business_action": (
            "Monitor antenatal care, skilled attendance and "
            "institutional-birth indicators together."
        )
    },
    {
        "priority": "Medium",
        "area": "Child nutrition",
        "finding": (
            "Maternal Care and Child Stunting show a moderate "
            "negative association (r = -0.360)."
        ),
        "recommendation": (
            "Integrate maternal-health and child-nutrition "
            "monitoring when identifying vulnerable districts."
        ),
        "business_action": (
            "Flag districts where maternal-care indicators and "
            "child-nutrition outcomes are simultaneously weak."
        )
    },
    {
        "priority": "Operational",
        "area": "Data quality",
        "finding": (
            f"{(district_score_final['score_status'] == 'Low Coverage').sum()} "
            "districts have insufficient scoring-indicator coverage."
        ),
        "recommendation": (
            "Treat low-coverage district scores cautiously and "
            "improve indicator completeness before strong comparisons."
        ),
        "business_action": (
            "Use score_coverage_pct as a quality-control field "
            "in reporting and dashboard interpretation."
        )
    }
])

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert recommendations["recommendation"].notna().all()
assert recommendations["finding"].notna().all()
assert recommendations["business_action"].notna().all()
assert recommendations["priority"].notna().all()

print("=" * 65)
print("RECOMMENDATIONS")
print("=" * 65)

print("Recommendations :", len(recommendations))
print("Priority levels :", recommendations["priority"].nunique())
print("Areas covered   :", recommendations["area"].nunique())
print("Missing cells   :", recommendations.isna().sum().sum())

display(recommendations)

print("\nVALIDATION: PASSED")

RECOMMENDATIONS
Recommendations : 6
Priority levels : 3
Areas covered   : 6
Missing cells   : 0


,priority,area,finding,recommendation,business_action
0,High,District-level healthcare,The lowest-ranked reliable districts have substantially larger HealthGap values.,Prioritize targeted healthcare interventions in the lowest-performing districts.,Use district ranking and HealthGap gap to prioritize locations for resource allocation.
1,High,Within-state disparity,329 reliable districts perform below their respective State/UT average.,Use district-level performance rather than state averages alone when identifying priority areas.,Investigate districts with the largest negative district_vs_state values.
2,High,Bihar,Bihar has the lowest average HealthGap score among the States/UTs analyzed.,Conduct an indicator-level assessment of Bihar to identify the main contributors to the observed...,Prioritize Bihar for deeper diagnostic analysis before allocating interventions.
3,Medium,Maternal care,Maternal Care and Institutional Births show a strong positive association (r = 0.599).,Strengthen maternal-care pathways that support access to institutional delivery.,"Monitor antenatal care, skilled attendance and institutional-birth indicators together."
4,Medium,Child nutrition,Maternal Care and Child Stunting show a moderate negative association (r = -0.360).,Integrate maternal-health and child-nutrition monitoring when identifying vulnerable districts.,Flag districts where maternal-care indicators and child-nutrition outcomes are simultaneously weak.
5,Operational,Data quality,13 districts have insufficient scoring-indicator coverage.,Treat low-coverage district scores cautiously and improve indicator completeness before strong c...,Use score_coverage_pct as a quality-control field in reporting and dashboard interpretation.



VALIDATION: PASSED


In [31]:
# ============================================================
# FINAL ANALYTICAL EXPORTS
# ============================================================

EXPORTS = {
    "healthgap_core_dataset.csv": df_core,
    "healthgap_supporting_dataset.csv": df_supporting,
    "healthgap_data_dictionary.csv": data_dictionary,
    "indicator_master.csv": indicator_master,
    "healthgap_kpi_master.csv": kpi_master,
    "healthgap_district_scores.csv": district_score_final,
    "healthgap_state_summary.csv": state_summary,
    "healthgap_priority_districts.csv": priority_districts,
    "healthgap_relationships.csv": relationship_df,
    "healthgap_state_opportunities.csv": state_opportunities,
    "healthgap_recommendations.csv": recommendations
}

for filename, data in EXPORTS.items():
    data.to_csv(filename, index=False)

print("=" * 65)
print("FINAL PROJECT EXPORT")
print("=" * 65)

for filename, data in EXPORTS.items():
    print(f"{filename:<40} {data.shape}")

FINAL PROJECT EXPORT
healthgap_core_dataset.csv               (706, 52)
healthgap_supporting_dataset.csv         (706, 51)
healthgap_data_dictionary.csv            (107, 6)
indicator_master.csv                     (107, 12)
healthgap_kpi_master.csv                 (50, 8)
healthgap_district_scores.csv            (706, 9)
healthgap_state_summary.csv              (36, 8)
healthgap_priority_districts.csv         (10, 6)
healthgap_relationships.csv              (2, 5)
healthgap_state_opportunities.csv        (36, 6)
healthgap_recommendations.csv            (6, 5)
